In [0]:
# ================================================================
# PHASE 18 — DATABRICKS APP
# ================================================================

print("=" * 70)
print("PHASE 18 — DATABRICKS APP")
print("=" * 70)

print()
print("Initializing GenAI Data Analyst Copilot application.")

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/05_api_serving.py

In [0]:
# ================================================================
# CELL 3 — APP DEPENDENCY CHECK
# ================================================================

print("=" * 70)
print("APP DEPENDENCY CHECK")
print("=" * 70)

required_functions = [
    "api_request",
    "ask_copilot",
    "format_copilot_response"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:

        failed_dependencies.append(
            function_name
        )

print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)

if failed_dependencies:

    raise RuntimeError(
        "Databricks App cannot continue. "
        "Missing functions: "
        + ", ".join(
            failed_dependencies
        )
    )

print()
print("App dependency check: PASS")

In [0]:
# ================================================================
# CELL 4 — GRADIO IMPORT
# ================================================================

try:

    import gradio as gr

    print(
        "Gradio version:",
        gr.__version__
    )

    print(
        "Gradio import: PASS"
    )

except Exception as e:

    raise RuntimeError(
        "Gradio import failed: "
        + str(e)
    )

In [0]:
# ================================================================
# CELL 5 — APPLICATION REQUEST HANDLER
# ================================================================

def app_request(question):
    """
    Handle a user question from the Databricks App.
    """

    if question is None:

        return (
            "Please enter a question.",
            "",
            "",
            "ERROR"
        )

    if not isinstance(question, str):

        return (
            "Question must be text.",
            "",
            "",
            "ERROR"
        )

    question = question.strip()

    if not question:

        return (
            "Please enter a question.",
            "",
            "",
            "ERROR"
        )

    try:

        response = api_request(
            question
        )

        if not response.get(
            "success",
            False
        ):

            return (
                str(
                    response.get(
                        "error",
                        "Copilot request failed."
                    )
                ),
                response.get(
                    "route",
                    ""
                ),
                "",
                "FAILED"
            )

        answer = response.get(
            "answer"
        )

        route = response.get(
            "route",
            ""
        )

        sources = response.get(
            "sources",
            []
        )

        execution_time = response.get(
            "execution_time_ms"
        )

        return (
            str(answer),
            str(route),
            str(sources),
            f"SUCCESS — {execution_time} ms"
        )

    except Exception as e:

        return (
            f"{type(e).__name__}: {str(e)}",
            "",
            "",
            "ERROR"
        )


print(
    "app_request(): PASS"
)

In [0]:
# ================================================================
# CELL 6 — APPLICATION HANDLER TEST
# ================================================================

print("=" * 70)
print("APPLICATION HANDLER TEST")
print("=" * 70)

test_question = (
    "What is the discount policy?"
)

result = app_request(
    test_question
)

print()
print("ANSWER:")
print(result[0])

print()
print("ROUTE:")
print(result[1])

print()
print("SOURCES:")
print(result[2])

print()
print("STATUS:")
print(result[3])

assert result[3].startswith(
    "SUCCESS"
)

print()
print("Application handler: PASS")

In [0]:
# ================================================================
# CELL 7 — BUILD DATABRICKS APP UI
# ================================================================

with gr.Blocks(
    title="GenAI Data Analyst Copilot"
) as app:

    gr.Markdown(
        """
        # 🤖 GenAI Data Analyst Copilot

        Ask questions about business data, sales,
        policies, and regional performance.

        The copilot automatically routes your question
        through SQL, RAG, or Hybrid processing.
        """
    )

    with gr.Row():

        question = gr.Textbox(
            label="Your Question",
            placeholder=(
                "Example: Which region generated "
                "the highest revenue?"
            ),
            lines=3,
            scale=4
        )

        submit = gr.Button(
            "Ask Copilot",
            variant="primary",
            scale=1
        )

    answer = gr.Textbox(
        label="Copilot Answer",
        lines=10
    )

    with gr.Row():

        route = gr.Textbox(
            label="Route"
        )

        status = gr.Textbox(
            label="Status"
        )

    sources = gr.Textbox(
        label="Sources",
        lines=6
    )

    submit.click(
        fn=app_request,
        inputs=question,
        outputs=[
            answer,
            route,
            sources,
            status
        ]
    )

    question.submit(
        fn=app_request,
        inputs=question,
        outputs=[
            answer,
            route,
            sources,
            status
        ]
    )


print(
    "Databricks App UI: PASS"
)

In [0]:
# ================================================================
# CELL 8 — APP VALIDATION
# ================================================================

print("=" * 70)
print("DATABRICKS APP VALIDATION")
print("=" * 70)

assert app is not None

assert callable(
    app_request
)

print(
    "Application object: PASS"
)

print(
    "Request handler: PASS"
)

print()
print(
    "Databricks App validation: PASS"
)

In [0]:
# ================================================================
# CELL 9 — ROUTE TESTING
# ================================================================

print("=" * 70)
print("DATABRICKS APP ROUTE TESTING")
print("=" * 70)

tests = [
    (
        "SQL",
        "Which region generated the highest revenue?"
    ),
    (
        "RAG",
        "What is the discount policy?"
    ),
    (
        "HYBRID",
        "Which region generated the highest revenue "
        "and what discount policy applies there?"
    )
]

passed = 0

for expected_route, question_text in tests:

    print()
    print(
        "Expected route:",
        expected_route
    )

    result = app_request(
        question_text
    )

    print(
        "Actual route:",
        result[1]
    )

    print(
        "Status:",
        result[3]
    )

    if (
        result[3].startswith("SUCCESS")
        and result[1].lower() == expected_route.lower()
    ):

        print(
            f"{expected_route}: PASS"
        )

        passed += 1

    else:

        print(
            f"{expected_route}: FAIL"
        )


print()
print(
    "Total tests:",
    len(tests)
)

print(
    "Passed:",
    passed
)

print(
    "Failed:",
    len(tests) - passed
)

assert passed == len(tests)

print()
print(
    "Route testing: PASS"
)

In [0]:
# ================================================================
# CELL 10 — PHASE 18 VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 18 — DATABRICKS APP VALIDATION")
print("=" * 70)

checks = [
    (
        "API available",
        callable(api_request)
    ),
    (
        "Copilot available",
        callable(ask_copilot)
    ),
    (
        "Response formatter available",
        callable(format_copilot_response)
    ),
    (
        "Gradio available",
        gr is not None
    ),
    (
        "Application handler available",
        callable(app_request)
    ),
    (
        "Application created",
        app is not None
    )
]

failed = 0

for name, passed_check in checks:

    print(
        f"{'PASS' if passed_check else 'FAIL'} - "
        f"{name}"
    )

    if not passed_check:
        failed += 1


print()
print(
    "Total checks:",
    len(checks)
)

print(
    "Failed checks:",
    failed
)

if failed == 0:

    print()
    print(
        "PHASE 18 STATUS: PASS ✓"
    )

else:

    print()
    print(
        "PHASE 18 STATUS: FAIL ✗"
    )

    raise RuntimeError(
        "Phase 18 validation failed."
    )

In [0]:
# ================================================================
# CELL 11 — LOCAL APPLICATION LAUNCH
# ================================================================

print("=" * 70)
print("LAUNCHING DATABRICKS APP")
print("=" * 70)

app.launch(
    share=False
)